# Train Reader ViT5 trên Colab — Giai đoạn 4 (R1 / R2)

1. Runtime → Change runtime type → **T4 GPU** (L4/A100 nhanh hơn và có bf16).
2. Trên máy local tạo zip (kèm file `COMMIT`):
   `git archive --add-virtual-file=COMMIT:$(git rev-parse --short HEAD) -o hmtt.zip HEAD src configs data/processed notebooks requirements.txt`
   rồi đặt vào Drive: `MyDrive/HMTT/hmtt.zip` (ghi đè bản cũ).
3. Sửa `RUN` ở ô thứ 3 (`'r1'` hoặc `'r2'`), rồi Runtime → **Run all**. Mỗi lần chạy một run.

- **R1**: train trên đoạn văn đúng (9.394 mẫu). **R2**: thêm 1 đoạn nhiễu top-5 của retriever cho mỗi câu trả lời được (~15.700 mẫu, lâu hơn ~1,7 lần).
- Thời gian chưa đo; ước tính R1 vài giờ trên T4 (5 epoch, có early stopping). Đừng để tab ngủ quá lâu, Colab miễn phí có thể ngắt.
- Nếu log báo **Loss NaN** (T5 + fp16 trên T4): đặt `EXTRA = '--no_fp16'` rồi Run all lại.
- Kết quả tự lưu vào `MyDrive/HMTT/predictions/` và `MyDrive/HMTT/models/{RUN}_vit5`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/HMTT'
!rm -rf /content/HMTT && mkdir -p /content/HMTT && cd /content/HMTT && unzip -q $DRIVE/hmtt.zip
%cd /content/HMTT
!cat COMMIT
import subprocess
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True)
print(gpu.stdout or gpu.stderr)
# Lệnh `!` lỗi KHÔNG dừng Run all -> kiểm GPU bằng assert Python để dừng hẳn
assert gpu.returncode == 0, 'KHÔNG có GPU: Runtime -> Change runtime type -> T4 GPU, rồi Run all lại'

In [ ]:
# Colab đã có torch khớp CUDA -> chỉ cài phần còn lại theo bản ghim
!pip install -q -r requirements.txt
!python -c "import transformers, torch; print(transformers.__version__, torch.__version__, torch.cuda.is_available()); assert transformers.__version__ == '4.57.6'; assert torch.cuda.is_available(), 'Chưa bật GPU'"

In [ ]:
RUN = 'r1'      # 'r1' = đoạn đúng | 'r2' = đoạn đúng + đoạn nhiễu của retriever
EXTRA = ''      # vd. '--no_fp16' nếu loss NaN; '--lr 3e-4' để thử lr khác
import os
assert RUN in ('r1', 'r2')
MODEL = f'models/{RUN}_vit5'

In [ ]:
# Train (mỗi epoch chấm dev trên đoạn đúng, giữ checkpoint F1 cao nhất)
!PYTHONIOENCODING=utf-8 python src/reader/train_seq2seq.py --data $RUN $EXTRA 2>&1 | grep -v Warning
assert os.path.exists(f'{MODEL}/train_config.json'), 'Train lỗi hoặc dừng giữa chừng — xem log phía trên'
# Lưu model lên Drive ngay (phòng khi bị ngắt ở bước suy luận)
!mkdir -p $DRIVE/models && rm -rf $DRIVE/$MODEL && cp -r $MODEL $DRIVE/models/
!ls -la $DRIVE/$MODEL

In [ ]:
# Suy luận trên dev: đoạn đúng, và đoạn retriever (hybrid top-3, dùng cache truy hồi)
!PYTHONIOENCODING=utf-8 python src/reader/infer.py --model $MODEL --context gold --run {RUN}_gold_dev 2>&1 | grep -v Warning
!PYTHONIOENCODING=utf-8 python src/reader/infer.py --model $MODEL --context retrieved --k 3 --run {RUN}_retrieved_k3_dev 2>&1 | grep -v Warning
!PYTHONIOENCODING=utf-8 python src/eval/evaluate_pipeline.py data/predictions/{RUN}_gold_dev.jsonl data/predictions/{RUN}_retrieved_k3_dev.jsonl

In [ ]:
!mkdir -p $DRIVE/predictions
!cp data/predictions/*.jsonl $DRIVE/predictions/
!ls -la $DRIVE/predictions